In [449]:
import pandas as pd
import numpy as np
import time
import faiss
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering, Birch, MiniBatchKMeans, KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.decomposition import NMF
import hdbscan

OPIS SKUPA PODATAKA

The dataset consists of 10 numerical and 8 categorical attributes.
The 'Revenue' attribute can be used as the class label.

"Administrative", "Administrative Duration", "Informational", "Informational Duration", "Product Related" and "Product Related Duration" represent the number of different types of pages visited by the visitor in that session and total time spent in each of these page categories. The values of these features are derived from the URL information of the pages visited by the user and updated in real time when a user takes an action, e.g. moving from one page to another. The "Bounce Rate", "Exit Rate" and "Page Value" features represent the metrics measured by "Google Analytics" for each page in the e-commerce site. The value of "Bounce Rate" feature for a web page refers to the percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. The value of "Exit Rate" feature for a specific web page is calculated as for all pageviews to the page, the percentage that were the last in the session. The "Page Value" feature represents the average value for a web page that a user visited before completing an e-commerce transaction. The "Special Day" feature indicates the closeness of the site visiting time to a specific special day (e.g. Mother’s Day, Valentine's Day) in which the sessions are more likely to be finalized with transaction. The value of this attribute is determined by considering the dynamics of e-commerce such as the duration between the order date and delivery date. For example, for Valentina’s day, this value takes a nonzero value between February 2 and February 12, zero before and after this date unless it is close to another special day, and its maximum value of 1 on February 8. The dataset also includes operating system, browser, region, traffic type, visitor type as returning or new visitor, a Boolean value indicating whether the date of the visit is weekend, and month of the year.

In [450]:
exel = []

In [451]:
def plot_clusters_pca(X, labels, title):
    X = np.array(X)
    if X.ndim == 1:
        raise ValueError("Input data X must be 2D for plotting clusters.")
    if X.shape[1] < 2:
        raise ValueError("Input data X must have at least two features (columns) for 2D plotting.")
    print(title,"Silhouette:", silhouette_score(X, labels) if len(set(labels)) > 1 else "N/A")
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], cmap='tab10', s=10)
    plt.title(title)
    plt.grid(True)
    plt.show()

In [452]:
def plot_clusters(X, labels, title):
    X = np.array(X)
    if X.ndim == 1:
        raise ValueError("Input data X must be 2D for plotting clusters.")
    if X.shape[1] < 2:
        raise ValueError("Input data X must have at least two features (columns) for 2D plotting.")
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    # print(title,"Silhouette:", silhouette_score(X, labels) if len(set(labels)) > 1 else "N/A")
    plt.figure(figsize=(6, 5))
    # plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=10)
    plt.scatter(X_pca[:, 0], X_pca[:, 1], cmap='tab10', s=10)
    plt.title(title)
    plt.grid(True)
    plt.show()

In [453]:
def metrike(title,X,labels,y):
    print("Silhouette +",title,":", silhouette_score(X, labels))
    print("Davies-Bouldin indeks: +",title,":", davies_bouldin_score(X, labels))
    ct = pd.crosstab(labels, y, normalize='index')
    print("Raspodjela ciljne po klasterima:")
    print(ct)


In [454]:
def metrike1(title,X,labels,y):
    ct = pd.crosstab(labels, y, normalize='index')
    return {
        "model": title,
        "Silhouette": round(silhouette_score(X, labels), 3),
        "Davies-Bouldin indeks": round(davies_bouldin_score(X, labels), 3),
        "Raspodjela ciljne po klasterima": ct,
    }

In [455]:
def cluster1(model, title, X,y):
    start = time.perf_counter()
    model.fit(X)
    

    labels = model.labels_
    met = metrike1(title, X, labels,y)
    print(met)
    end = time.perf_counter()
    trajanje = end - start
    exel.append(met)
    print(f"{title} | fit trajao: {trajanje:.4f} s")
    print("*"*50)

In [456]:
def cluster2(model, title, X,y):
    start = time.perf_counter()
    model.fit_predict(X)
    

    labels = model.labels_
    met = metrike1(title, X, labels,y)
    print(met)
    end = time.perf_counter()
    trajanje = end - start
    exel.append(met)
    print(f"{title} | fit trajao: {trajanje:.4f} s")
    print("*"*50)

In [457]:
def faisscluster(title, X, i,y):
    start = time.perf_counter()

    faiss_kmeans = faiss.Kmeans(d=X.shape[1], k=i)
    faiss_kmeans.train(X)

    D, I = faiss_kmeans.index.search(X, 1)
    labels_faiss = I.flatten()
    met = metrike1(title, X, labels_faiss, y)
    print(met)

    end = time.perf_counter()
    trajanje = end - start
    exel.append(met)
    print(f"{title} | fit trajao: {trajanje:.4f} s")
    print("*"*50)

In [458]:
df = pd.read_csv(r'podaci\online+shoppers+purchasing+intention+dataset\online_shoppers_intention preprocessed.csv', encoding='cp1252', sep=',')
print(df.columns)

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='object')


In [459]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,1,1,1,1,2,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,2,1,2,2,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,4,1,9,3,2,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,2,3,2,2,4,2,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,2,3,3,1,4,2,1,0


In [460]:
y = df['Revenue']

In [461]:
X = df.copy(deep=True)

In [462]:
X = X.drop('Revenue', axis=1)

In [463]:
engagement_features = ["Administrative","Administrative_Duration","Informational","Informational_Duration","ProductRelated",
        "ProductRelated_Duration","BounceRates","ExitRates","PageValues"]
technical_features = ["OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType"]
time_features = ["SpecialDay","Month","Weekend"]

In [464]:
numeric_features = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues"
]

In [465]:
pca = PCA(n_components=2)
scaler_minmax = MinMaxScaler()
scaler_standard = StandardScaler()

In [466]:
X_log = X.copy(deep=True)

In [467]:
X_log[numeric_features] = np.log1p(X_log[numeric_features])     # Logaritamsko skaliranje
X_sca_std = scaler_standard.fit_transform(X)                    # Standard scaler
X_sca_mm = scaler_minmax.fit_transform(X)                       # MinMax scaler

X_pca = pca.fit_transform(X)

X_pca_sca_std = pca.fit_transform(X_sca_std)                    # Standard scaler + PCA
X_pca_sca_mm = pca.fit_transform(X_sca_mm)                      # MinMax scaler + PCA

In [468]:
datasets = [
    # ("Originalni podaci", X),
    # ("Log transformacija", X_log),
    # ("StandardScaler", X_sca_std),
    # ("MinMaxScaler", X_sca_mm),
    # ("PCA", X_pca),
    # ("StandardScaler + PCA", X_pca_sca_std),
    ("MinMaxScaler + PCA", X_pca_sca_mm),
]

In [469]:
# cols = X.columns
# n_cols = 2
# n_rows = len(cols)
# scaled_df_std = pd.DataFrame(X_sca_std, columns=X.columns, index=X.index)
# scaled_df_mm = pd.DataFrame(X_sca_mm, columns=X.columns, index=X.index)

In [470]:
# cols_to_plot = ["Administrative","Administrative_Duration","Informational","Informational_Duration","ProductRelated",
#         "ProductRelated_Duration","BounceRates","ExitRates","PageValues"]

# for col in cols_to_plot:
#     plt.figure(figsize=(10,4))
#     plt.suptitle(f'{col}')

#     # Prije
#     plt.subplot(1,3,1)
#     sns.histplot(scaled_df_std[col], kde=True, color="skyblue")
#     plt.title(f"Standardno skaliranje")


#     # Poslije
#     plt.subplot(1,3,3)
#     sns.histplot(X_log[col], kde=True, color="salmon")
#     plt.title(f"Logaritmsko sklairanje")

#     plt.subplot(1,3,2)
#     sns.histplot(scaled_df_mm[col], kde=True, color="green")
#     plt.title(f"MinMax skaliranje")

#     plt.tight_layout()
#     plt.show()

In [471]:
for i in range(2,3):
    mb_kmeans = MiniBatchKMeans(n_clusters=i, random_state=123)
    print("----------------"*2," Broj klastera ",i,"--------------------"*2)

    for name, X in datasets:
        cluster1(mb_kmeans,'MiniBatchKMeans', X, y)

--------------------------------  Broj klastera  2 ----------------------------------------
{'model': 'MiniBatchKMeans', 'Silhouette': np.float64(0.634), 'Davies-Bouldin indeks': np.float64(0.588), 'Raspodjela ciljne po klasterima': Revenue         0         1
row_0                      
0        0.826011  0.173989
1        0.851089  0.148911}
MiniBatchKMeans | fit trajao: 1.3285 s
**************************************************


In [472]:
for i in range(2,3):
    spectral = SpectralClustering(n_clusters=i, affinity='nearest_neighbors', random_state=123)

    print("----------------"*2," Broj klastera ",i,"--------------------"*2)
    
    for name, X in datasets:
            cluster2(spectral, 'SpectralClustering', X, y)

--------------------------------  Broj klastera  2 ----------------------------------------


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


{'model': 'SpectralClustering', 'Silhouette': np.float64(0.634), 'Davies-Bouldin indeks': np.float64(0.588), 'Raspodjela ciljne po klasterima': Revenue         0         1
row_0                      
0        0.826011  0.173989
1        0.851089  0.148911}
SpectralClustering | fit trajao: 31.0335 s
**************************************************


In [473]:
for i in range(2,3):
    kmeans = KMeans(n_clusters=i , init='k-means++', random_state=123)
    
    print("----------------"*2," Broj klastera ",i,"--------------------"*2)
    
    for name, X in datasets:
            cluster2(kmeans,'KMeans',X,y)

--------------------------------  Broj klastera  2 ----------------------------------------
{'model': 'KMeans', 'Silhouette': np.float64(0.634), 'Davies-Bouldin indeks': np.float64(0.588), 'Raspodjela ciljne po klasterima': Revenue         0         1
row_0                      
0        0.826011  0.173989
1        0.851089  0.148911}
KMeans | fit trajao: 1.2764 s
**************************************************


In [474]:
for i in range(2,3):
    birch = Birch(n_clusters=i)
    
    print("----------------"*2," Broj klastera ",i,"--------------------"*2)
    
    for name, X in datasets:
            cluster1(birch,'Birch',X,y)

--------------------------------  Broj klastera  2 ----------------------------------------
{'model': 'Birch', 'Silhouette': np.float64(0.634), 'Davies-Bouldin indeks': np.float64(0.588), 'Raspodjela ciljne po klasterima': Revenue         0         1
row_0                      
0        0.826011  0.173989
1        0.851089  0.148911}
Birch | fit trajao: 1.3844 s
**************************************************


In [475]:
for i in range(2,3):
    print("----------------"*2," Broj klastera ",i,"--------------------"*2)

    for name, X in datasets:
            faisscluster('Faiss',X,i,y)

--------------------------------  Broj klastera  2 ----------------------------------------
{'model': 'Faiss', 'Silhouette': np.float64(0.335), 'Davies-Bouldin indeks': np.float64(1.448), 'Raspodjela ciljne po klasterima': Revenue         0         1
row_0                      
0        0.777313  0.222687
1        0.914309  0.085691}
Faiss | fit trajao: 1.2580 s
**************************************************


In [476]:
hdb = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=10)

for i in range(2,3):
    print("----------------"*2," Broj klastera ",i,"--------------------"*2)

    for name, X in datasets:
            cluster1(hdb,'HDBSCAN',X,y)

--------------------------------  Broj klastera  2 ----------------------------------------
{'model': 'HDBSCAN', 'Silhouette': np.float64(0.218), 'Davies-Bouldin indeks': np.float64(1.033), 'Raspodjela ciljne po klasterima': Revenue         0         1
row_0                      
-1       0.910861  0.089139
 0       1.000000  0.000000
 1       1.000000  0.000000
 2       0.804598  0.195402
 3       0.757895  0.242105
 4       1.000000  0.000000
 5       0.988571  0.011429
 6       0.629630  0.370370
 7       1.000000  0.000000
 8       0.853034  0.146966
 9       1.000000  0.000000
 10      1.000000  0.000000
 11      1.000000  0.000000
 12      0.713246  0.286754
 13      0.747368  0.252632
 14      1.000000  0.000000
 15      0.991870  0.008130
 16      0.849078  0.150922
 17      0.977273  0.022727
 18      0.800000  0.200000
 19      0.818182  0.181818
 20      0.759756  0.240244}
HDBSCAN | fit trajao: 1.3612 s
**************************************************


In [477]:
for i in range(2,3):
    agg = AgglomerativeClustering(n_clusters=i)

    print("---------------- Broj klastera ",i,"--------------------")

    for name, X in datasets:
            cluster2(agg,'AGG',X,y)

---------------- Broj klastera  2 --------------------
{'model': 'AGG', 'Silhouette': np.float64(0.634), 'Davies-Bouldin indeks': np.float64(0.588), 'Raspodjela ciljne po klasterima': Revenue         0         1
row_0                      
0        0.851089  0.148911
1        0.826011  0.173989}
AGG | fit trajao: 3.7047 s
**************************************************


In [478]:
df_exel = pd.DataFrame(exel)
df_exel.to_excel("rezultatiKlasterOSI/rez_pca_sca_mm.xlsx", index=False)

In [479]:
# for name, X in datasets:

#     linked = linkage(X, method='ward')

#     plt.figure(figsize=(12, 6))
#     dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=False)
#     plt.title(f'Dendogram {name}')
#     plt.show()